# 30_gold_dims — Gold Dimension Tables

Builds all Kimball-style dimensions. **Must run before `31_gold_fact`**.

| Table | Strategy |
|---|---|
| `gold.dim_date` | Overwrite (generated) |
| `gold.dim_time` | Overwrite (24 rows) |
| `gold.dim_device` | MERGE SCD1 + -1 Unknown |
| `gold.dim_track` | MERGE SCD1 + inferred members |
| `gold.dim_user` | SCD2 MERGE |

In [ ]:
%run ./00_config


## Step 1: dim_date — Generated calendar dimension

In [ ]:
from datetime import datetime, timezone

_t0 = datetime.now(timezone.utc)
try:
    bounds = spark.sql(
        f'SELECT MIN(_partition_dt) AS mn, DATE_ADD(MAX(_partition_dt), 365) AS mx'
        f' FROM {table_ref(SILVER_EVENTS)}'
    ).first()
    if bounds['mn'] is None:
        raise ValueError('silver.events is empty')
    mn, mx = str(bounds['mn']), str(bounds['mx'])

    df_date = spark.sql(f"""
        SELECT
            CAST(DATE_FORMAT(d,'yyyyMMdd') AS INT) AS date_key,
            d                                       AS date,
            YEAR(d)                                 AS year,
            QUARTER(d)                              AS quarter,
            MONTH(d)                                AS month,
            DATE_FORMAT(d,'MMMM')                  AS month_name,
            WEEKOFYEAR(d)                           AS week_of_year,
            DAYOFWEEK(d)                            AS day_of_week,
            DATE_FORMAT(d,'EEEE')                  AS day_name,
            CASE WHEN DAYOFWEEK(d) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM (SELECT EXPLODE(SEQUENCE(DATE '{mn}', DATE '{mx}', INTERVAL 1 DAY)) AS d)
    """)
    n = df_date.count()
    df_date.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['dim_date'])
    log_run('SILVER_TO_GOLD','gold.dim_date',f'{mn}..{mx}',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'dim_date: {n:,} rows ({mn} to {mx})')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.dim_date',START_DATE,_t0,'FAILURE',error_message=str(_e))
    raise


## Step 2: dim_time — 24-row hour dimension

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_time = spark.sql("""
        SELECT
            CAST(h AS TINYINT) AS hour_of_day,
            CASE WHEN h = 0  THEN '12AM'
                 WHEN h < 12 THEN LPAD(CAST(h AS STRING),2,'0')||'AM'
                 WHEN h = 12 THEN '12PM'
                 ELSE LPAD(CAST(h-12 AS STRING),2,'0')||'PM' END AS hour_label,
            CASE WHEN h BETWEEN 0  AND 5  THEN 'Night'
                 WHEN h BETWEEN 6  AND 11 THEN 'Morning'
                 WHEN h BETWEEN 12 AND 17 THEN 'Afternoon'
                 ELSE 'Evening' END AS day_part,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM (SELECT EXPLODE(SEQUENCE(0,23)) AS h)
    """)
    df_time.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['dim_time'])
    log_run('SILVER_TO_GOLD','gold.dim_time','static-24-rows',_t0,'SUCCESS',
            rows_read=24,rows_inserted=24)
    print('dim_time: 24 rows written')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.dim_time','static-24-rows',_t0,'FAILURE',error_message=str(_e))
    raise


## Step 3: dim_device — SCD1 + Unknown member

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    rows = [(dt, cat) for dt, cat in DEVICE_CATEGORY.items()] + [('unknown','unknown')]
    df_dev = spark.createDataFrame(rows, ['device_type','device_category']) \
        .withColumn('device_sk',
            F.when(F.col('device_type')=='unknown', F.lit(UNKNOWN_SK).cast('bigint'))
             .otherwise(sk(F.col('device_type')))) \
        .withColumn('updated_at',     F.current_timestamp()) \
        .withColumn('load_timestamp', F.current_timestamp())
    n = df_dev.count()
    df_dev.createOrReplaceTempView('_dev_stage')
    spark.sql(f"""
        MERGE INTO {table_ref(GOLD['dim_device'])} AS tgt
        USING _dev_stage AS src ON tgt.device_sk = src.device_sk
        WHEN MATCHED AND tgt.device_category <> src.device_category THEN
            UPDATE SET tgt.device_category=src.device_category,
                       tgt.updated_at=src.updated_at,
                       tgt.load_timestamp=src.load_timestamp
        WHEN NOT MATCHED THEN
            INSERT (device_sk,device_type,device_category,created_at,updated_at,load_timestamp)
            VALUES (src.device_sk,src.device_type,src.device_category,
                    src.updated_at,src.updated_at,src.load_timestamp)
    """)
    m = execution_metrics(GOLD['dim_device'])
    log_run('SILVER_TO_GOLD','gold.dim_device','SCD1',_t0,'SUCCESS',
            rows_read=n,rows_inserted=m['inserted'],rows_updated=m['updated'])
    print(f'dim_device: {n} src | +{m["inserted"]} ins, ~{m["updated"]} upd')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.dim_device','SCD1',_t0,'FAILURE',error_message=str(_e))
    raise


## Step 4: dim_track — SCD1 + inferred members for early-arriving facts

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    ts = F.current_timestamp()
    rh = F.md5(F.concat_ws('||', F.col('track_name'), F.col('artist'),
                            F.col('genre'), F.col('release_year').cast('string'),
                            F.col('duration_sec').cast('string'),
                            F.col('popularity').cast('string')))
    df_t = spark.table(SILVER_TRACKS).withColumns({
        'track_sk':     sk(F.col('track_id')),
        'artist_name':  F.col('artist'),
        'is_inferred':  F.lit(False),
        '_row_hash':    rh,
        'created_at':   ts,
        'updated_at':   ts,
        'load_timestamp': ts,
    }).select('track_sk','track_id','track_name','artist_name','genre',
               'release_year','duration_sec','popularity',
               'danceability','energy','valence','tempo_bpm',
               'is_inferred','_row_hash','created_at','updated_at','load_timestamp')
    n = df_t.count()
    df_t.createOrReplaceTempView('_trk_stage')

    spark.sql(f"""
        MERGE INTO {table_ref(GOLD['dim_track'])} AS tgt
        USING _trk_stage AS src ON tgt.track_sk = src.track_sk
        WHEN MATCHED AND tgt._row_hash <> src._row_hash THEN
            UPDATE SET tgt.track_name=src.track_name, tgt.artist_name=src.artist_name,
                       tgt.genre=src.genre, tgt.release_year=src.release_year,
                       tgt.duration_sec=src.duration_sec, tgt.popularity=src.popularity,
                       tgt.danceability=src.danceability, tgt.energy=src.energy,
                       tgt.valence=src.valence, tgt.tempo_bpm=src.tempo_bpm,
                       tgt.is_inferred=FALSE, tgt._row_hash=src._row_hash,
                       tgt.updated_at=src.updated_at, tgt.load_timestamp=src.load_timestamp
        WHEN NOT MATCHED THEN
            INSERT (track_sk,track_id,track_name,artist_name,genre,release_year,
                    duration_sec,popularity,danceability,energy,valence,tempo_bpm,
                    is_inferred,_row_hash,created_at,updated_at,load_timestamp)
            VALUES (src.track_sk,src.track_id,src.track_name,src.artist_name,
                    src.genre,src.release_year,src.duration_sec,src.popularity,
                    src.danceability,src.energy,src.valence,src.tempo_bpm,
                    FALSE,src._row_hash,src.created_at,src.updated_at,src.load_timestamp)
    """)
    m1 = execution_metrics(GOLD['dim_track'])

    # Inferred members for early-arriving facts
    spark.sql(f"""
        MERGE INTO {table_ref(GOLD['dim_track'])} AS tgt
        USING (
            SELECT DISTINCT xxhash64(e.track_id) AS track_sk, e.track_id,
                'Unknown Track' AS track_name, 'Unknown Artist' AS artist_name,
                'unclassified' AS genre,
                CAST(NULL AS INT) AS release_year, CAST(NULL AS INT) AS duration_sec,
                CAST(NULL AS INT) AS popularity,
                CAST(NULL AS DOUBLE) AS danceability, CAST(NULL AS DOUBLE) AS energy,
                CAST(NULL AS DOUBLE) AS valence, CAST(NULL AS DOUBLE) AS tempo_bpm,
                TRUE AS is_inferred, MD5(e.track_id) AS _row_hash,
                CURRENT_TIMESTAMP() AS created_at, CURRENT_TIMESTAMP() AS updated_at,
                CURRENT_TIMESTAMP() AS load_timestamp
            FROM {table_ref(SILVER_EVENTS)} e
            WHERE NOT EXISTS (
                SELECT 1 FROM {table_ref(GOLD['dim_track'])} d WHERE d.track_id = e.track_id)
        ) AS inf ON tgt.track_sk = inf.track_sk
        WHEN NOT MATCHED THEN INSERT *
    """)
    m2 = execution_metrics(GOLD['dim_track'])

    log_run('SILVER_TO_GOLD','gold.dim_track','SCD1+inferred',_t0,'SUCCESS',
            rows_read=n, rows_inserted=m1['inserted']+m2['inserted'],
            rows_updated=m1['updated'])
    print(f'dim_track: {n} src | +{m1["inserted"]} known, ~{m1["updated"]} upd, +{m2["inserted"]} inferred')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.dim_track','SCD1+inferred',_t0,'FAILURE',error_message=str(_e))
    raise


## Step 5: dim_user — SCD2 MERGE

- **type2_hash** = `MD5(is_premium)` — triggers a new row version when it changes
- **type1_hash** = `MD5(country || timezone)` — overwrites in place
- `valid_from` for first version = `FIRST_VALID_FROM` (1900-01-01)
- Snapshot timestamp = `MAX(load_timestamp)` from `silver.users` (unchanged Silver → no new version)

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    snap_ts = spark.sql(
        f'SELECT MAX(load_timestamp) AS s FROM {table_ref(SILVER_USERS)}'
    ).first()['s']

    if snap_ts is None:
        log_run('SILVER_TO_GOLD','gold.dim_user','SCD2',_t0,'SKIPPED_UNCHANGED',rows_read=0)
        print('dim_user: silver.users empty — skipped')
    else:
        n_read = spark.table(SILVER_USERS).count()
        df_su = spark.table(SILVER_USERS).withColumns({
            'type2_hash':    F.md5(F.col('is_premium').cast('string')),
            'type1_hash':    F.md5(F.concat_ws('||', F.col('country'), F.col('timezone'))),
            'source_system': F.when(F.col('user_id').like('user_real_%'),
                                     F.lit('real')).otherwise(F.lit('synthetic')),
        })
        df_su.createOrReplaceTempView('_su_stage')

        dim_n = spark.sql(f'SELECT COUNT(1) AS c FROM {table_ref(GOLD["dim_user"])}').first()['c']

        if dim_n == 0:
            # ---- Initial load ----
            spark.sql(f"""
                INSERT INTO {table_ref(GOLD['dim_user'])}
                SELECT
                    xxhash64(CONCAT_WS('||', user_id, '{FIRST_VALID_FROM}')) AS user_sk,
                    user_id, source_system, country, timezone, is_premium,
                    CAST('{FIRST_VALID_FROM}' AS TIMESTAMP) AS valid_from,
                    CAST('{OPEN_ENDED}'       AS TIMESTAMP) AS valid_to,
                    TRUE AS is_current, type1_hash, type2_hash,
                    CURRENT_TIMESTAMP() AS created_at,
                    CURRENT_TIMESTAMP() AS updated_at,
                    CURRENT_TIMESTAMP() AS load_timestamp
                FROM _su_stage
            """)
            # Unknown member
            spark.sql(f"""
                MERGE INTO {table_ref(GOLD['dim_user'])} AS t
                USING (SELECT {UNKNOWN_SK}L AS user_sk) AS s ON t.user_sk = s.user_sk
                WHEN NOT MATCHED THEN
                    INSERT (user_sk,user_id,source_system,country,timezone,is_premium,
                            valid_from,valid_to,is_current,type1_hash,type2_hash,
                            created_at,updated_at,load_timestamp)
                    VALUES ({UNKNOWN_SK}L,'user_unknown','synthetic',NULL,NULL,NULL,
                            CAST('{FIRST_VALID_FROM}' AS TIMESTAMP),
                            CAST('{OPEN_ENDED}' AS TIMESTAMP),TRUE,NULL,NULL,
                            CURRENT_TIMESTAMP(),CURRENT_TIMESTAMP(),CURRENT_TIMESTAMP())
            """)
            m = execution_metrics(GOLD['dim_user'])
            log_run('SILVER_TO_GOLD','gold.dim_user',f'SCD2 initial {snap_ts}',_t0,'SUCCESS',
                    rows_read=n_read,rows_inserted=m['inserted'])
            print(f'dim_user initial load: {m["inserted"]:,} rows')

        else:
            # ---- Incremental SCD2 ----
            # Type-1: overwrite country / timezone in place
            spark.sql(f"""
                MERGE INTO {table_ref(GOLD['dim_user'])} AS tgt
                USING _su_stage AS src ON tgt.user_id=src.user_id AND tgt.is_current=TRUE
                WHEN MATCHED AND tgt.type1_hash <> src.type1_hash THEN
                    UPDATE SET tgt.country=src.country, tgt.timezone=src.timezone,
                               tgt.type1_hash=src.type1_hash,
                               tgt.updated_at=CURRENT_TIMESTAMP(),
                               tgt.load_timestamp=CURRENT_TIMESTAMP()
            """)
            # Type-2: expire current row where is_premium changed
            spark.sql(f"""
                MERGE INTO {table_ref(GOLD['dim_user'])} AS tgt
                USING _su_stage AS src ON tgt.user_id=src.user_id AND tgt.is_current=TRUE
                WHEN MATCHED AND tgt.type2_hash <> src.type2_hash THEN
                    UPDATE SET tgt.valid_to=CAST('{snap_ts}' AS TIMESTAMP),
                               tgt.is_current=FALSE,
                               tgt.updated_at=CURRENT_TIMESTAMP(),
                               tgt.load_timestamp=CURRENT_TIMESTAMP()
            """)
            # Insert new versions (type-2 changed) or brand-new users
            spark.sql(f"""
                MERGE INTO {table_ref(GOLD['dim_user'])} AS tgt
                USING (
                    SELECT xxhash64(CONCAT_WS('||',s.user_id,CAST('{snap_ts}' AS STRING))) AS user_sk,
                           s.user_id,s.source_system,s.country,s.timezone,s.is_premium,
                           CAST('{snap_ts}' AS TIMESTAMP) AS valid_from,
                           CAST('{OPEN_ENDED}' AS TIMESTAMP) AS valid_to,
                           TRUE AS is_current, s.type1_hash, s.type2_hash
                    FROM _su_stage s
                    WHERE NOT EXISTS (
                        SELECT 1 FROM {table_ref(GOLD['dim_user'])} d
                        WHERE d.user_id=s.user_id AND d.is_current=TRUE)
                ) AS nv ON tgt.user_sk = nv.user_sk
                WHEN NOT MATCHED THEN
                    INSERT (user_sk,user_id,source_system,country,timezone,is_premium,
                            valid_from,valid_to,is_current,type1_hash,type2_hash,
                            created_at,updated_at,load_timestamp)
                    VALUES (nv.user_sk,nv.user_id,nv.source_system,nv.country,nv.timezone,
                            nv.is_premium,nv.valid_from,nv.valid_to,nv.is_current,
                            nv.type1_hash,nv.type2_hash,
                            CURRENT_TIMESTAMP(),CURRENT_TIMESTAMP(),CURRENT_TIMESTAMP())
            """)
            m = execution_metrics(GOLD['dim_user'])
            log_run('SILVER_TO_GOLD','gold.dim_user',f'SCD2 incr {snap_ts}',_t0,'SUCCESS',
                    rows_read=n_read,rows_inserted=m['inserted'],rows_updated=m['updated'])
            print(f'dim_user incremental: +{m["inserted"]} new versions, ~{m["updated"]} upd')

except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.dim_user','SCD2',_t0,'FAILURE',error_message=str(_e))
    raise


## Proof: Gold Dimension Row Counts

In [ ]:
print('=== Gold Dimension Summary ===')
for dim in ['dim_date','dim_time','dim_device','dim_track','dim_user']:
    n = spark.table(GOLD[dim]).count()
    print(f'  {GOLD[dim]}: {n:,} rows')

print('\n--- dim_user: SCD2 version distribution ---')
display(spark.sql(f"""
    SELECT is_current, source_system, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users
    FROM {table_ref(GOLD['dim_user'])} WHERE user_id <> 'user_unknown'
    GROUP BY is_current, source_system ORDER BY is_current DESC
"""))

print('\n--- dim_track: inferred vs known by genre ---')
display(spark.sql(f"""
    SELECT is_inferred, COUNT(*) AS tracks
    FROM {table_ref(GOLD['dim_track'])}
    GROUP BY is_inferred
"""))
